# Estudo de Caso 5.2 — Profundidade Normal em Canal Trapezoidal

**Capítulo Associado:** Capítulo 5 — Hidrodinâmica de Canais Abertos  
**Nível:** Graduação  
**Tempo estimado:** 45 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, determinamos a profundidade normal ($h_n$) de um canal trapezoidal utilizando a equação de Manning. Como a equação é não-linear em $h$, resolvemos numericamente pelo método de Brent. O notebook inclui:

1. Definição da geometria trapezoidal
2. Implementação da equação de Manning
3. Solução numérica por método iterativo
4. Visualização da curva vazão-profundidade
5. Análise de sensibilidade aos parâmetros

---

## 🎯 Objetivos de Aprendizagem

- Calcular propriedades geométricas de seção trapezoidal
- Aplicar a equação de Manning para vazão
- Resolver equações não-lineares por métodos numéricos
- Interpretar a relação vazão-profundidade
- Analisar sensibilidade aos parâmetros hidráulicos

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✅ Ambiente configurado com sucesso!")
print(f"📦 NumPy {np.__version__} | SciPy {__import__('scipy').__version__}")

---

## 📐 Seção 1: Geometria do Canal Trapezoidal

Para um canal trapezoidal com largura de fundo $b$ e talude lateral $m:1$ (horizontal:vertical), as propriedades geométricas são:

$$
A(h) = (b + m \cdot h) \cdot h
$$

$$
P(h) = b + 2 \cdot h \cdot \sqrt{1 + m^2}
$$

$$
R_h(h) = \frac{A(h)}{P(h)}
$$

onde:
- $A$ = área molhada [m²]
- $P$ = perímetro molhado [m]
- $R_h$ = raio hidráulico [m]

In [ ]:
# ============================================================
# GEOMETRIA TRAPEZOIDAL
# ============================================================

def geometria_trapezoidal(b, m, h):
    """
    Calcula propriedades geométricas de seção trapezoidal.
    
    Parâmetros:
    -----------
    b : float
        Largura de fundo [m]
    m : float
        Inclinação do talude (H:V)
    h : float
        Profundidade da lâmina d'água [m]
    
    Retorna:
    --------
    dict
        {'A': área, 'P': perímetro, 'Rh': raio hidráulico, 'B': largura superficial}
    """
    A = (b + m * h) * h
    P = b + 2 * h * np.sqrt(1 + m**2)
    Rh = A / P
    B = b + 2 * m * h  # Largura da superfície livre
    
    return {'A': A, 'P': P, 'Rh': Rh, 'B': B}

# Parâmetros do canal
b = 3.0           # Largura de fundo [m]
m = 1.5           # Talude (H:V)
n = 0.025         # Coeficiente de Manning [s/m^(1/3)]
S0 = 0.001        # Declividade do fundo [m/m]
Q = 5.0           # Vazão [m³/s]

print("=" * 60)
print("PARÂMETROS DO CANAL TRAPEZOIDAL")
print("=" * 60)
print(f"\n📊 Geometria:")
print(f"  • Largura de fundo: b = {b} m")
print(f"  • Talude: m = {m} (H:V)")
print(f"\n🔧 Hidráulica:")
print(f"  • Manning: n = {n} s/m^(1/3)")
print(f"  • Declividade: S₀ = {S0}")
print(f"  • Vazão: Q = {Q} m³/s")

# Exemplo de cálculo para h = 1.0 m
h_exemplo = 1.0
geo = geometria_trapezoidal(b, m, h_exemplo)

print(f"\n📏 Propriedades para h = {h_exemplo} m:")
print(f"  • Área molhada: A = {geo['A']:.3f} m²")
print(f"  • Perímetro: P = {geo['P']:.3f} m")
print(f"  • Raio hidráulico: R_h = {geo['Rh']:.3f} m")
print(f"  • Largura superficial: B = {geo['B']:.3f} m")

---

## 🌊 Seção 2: Equação de Manning

A equação de Manning para vazão em canal é:

$$
Q = \frac{1}{n} \cdot A \cdot R_h^{2/3} \cdot S_0^{1/2}
$$

Para encontrar a profundidade normal $h_n$, precisamos resolver:

$$
f(h) = Q_{\text{Manning}}(h) - Q_{\text{dado}} = 0
$$

Esta é uma equação não-linear em $h$, pois $A(h)$ e $R_h(h)$ dependem de $h$.

In [ ]:
# ============================================================
# EQUAÇÃO DE MANNING
# ============================================================

def Q_Manning(b, m, n, S0, h):
    """
    Calcula a vazão pela equação de Manning.
    
    Parâmetros:
    -----------
    b, m, n, S0 : float
        Parâmetros do canal
    h : float
        Profundidade [m]
    
    Retorna:
    --------
    float
        Vazão [m³/s]
    """
    geo = geometria_trapezoidal(b, m, h)
    return (1/n) * geo['A'] * geo['Rh']**(2/3) * np.sqrt(S0)

def residual(b, m, n, S0, Q_dado, h):
    """
    Resíduo para encontrar a profundidade normal.
    
    f(h) = Q_Manning(h) - Q_dado
    """
    return Q_Manning(b, m, n, S0, h) - Q_dado

# Teste da função
h_teste = 1.0
Q_teste = Q_Manning(b, m, n, S0, h_teste)

print("=" * 60)
print("TESTE DA EQUAÇÃO DE MANNING")
print("=" * 60)
print(f"\n🧮 Para h = {h_teste} m:")
print(f"  • Q_Manning = {Q_teste:.3f} m³/s")
print(f"  • Q_dado = {Q} m³/s")
print(f"  • Resíduo = {Q_teste - Q:.3f} m³/s")

---

## 🔍 Seção 3: Solução Numérica (Método de Brent)

O método de Brent combina bisseção, secante e inversão quadrática para encontrar raízes de forma robusta e eficiente. É implementado na função `brentq` do SciPy.

In [ ]:
# ============================================================
# SOLUÇÃO NUMÉRICA
# ============================================================

print("=" * 60)
print("SOLUÇÃO NUMÉRICA - MÉTODO DE BRENT")
print("=" * 60)

# Definir função residual
def f(h):
    return residual(b, m, n, S0, Q, h)

# Intervalo de busca
h_min = 0.01
h_max = 10.0

# Verificar se há mudança de sinal
f_min = f(h_min)
f_max = f(h_max)

print(f"\n🔍 Intervalo de busca: [{h_min}, {h_max}] m")
print(f"  • f({h_min}) = {f_min:.3f}")
print(f"  • f({h_max}) = {f_max:.3f}")

if f_min * f_max < 0:
    print(f"\n✅ Há mudança de sinal → raiz existe no intervalo")
    
    # Resolver
    h_n = brentq(f, h_min, h_max)
    
    print(f"\n🎯 Profundidade normal:")
    print(f"  • h_n = {h_n:.4f} m")
    print(f"  • h_n = {h_n*100:.1f} cm")
    
    # Verificação
    Q_calc = Q_Manning(b, m, n, S0, h_n)
    print(f"\n✅ Verificação:")
    print(f"  • Q_Manning(h_n) = {Q_calc:.4f} m³/s")
    print(f"  • Q_dado = {Q} m³/s")
    print(f"  • Erro relativo = {abs(Q_calc - Q)/Q*100:.6f}%")
    
    # Propriedades geométricas
    geo = geometria_trapezoidal(b, m, h_n)
    print(f"\n📏 Propriedades geométricas:")
    print(f"  • Área: A = {geo['A']:.3f} m²")
    print(f"  • Perímetro: P = {geo['P']:.3f} m")
    print(f"  • Raio hidráulico: R_h = {geo['Rh']:.3f} m")
    print(f"  • Largura superficial: B = {geo['B']:.3f} m")
    
    # Velocidade média
    U = Q / geo['A']
    print(f"  • Velocidade média: U = {U:.3f} m/s")
    
else:
    print(f"\n❌ Não há mudança de sinal → ajustar intervalo")

---

## 📊 Seção 4: Visualização da Curva Vazão-Profundidade

Vamos plotar a curva $Q(h)$ para visualizar a relação não-linear e identificar a profundidade normal.

In [ ]:
# ============================================================
# VISUALIZAÇÃO
# ============================================================

# Gerar curva Q(h)
h_range = np.linspace(0.1, 3.0, 100)
Q_range = [Q_Manning(b, m, n, S0, h) for h in h_range]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(Q_range, h_range, 'b-', linewidth=2, label='Curva Q(h)')
ax.axvline(x=Q, color='r', linestyle='--', linewidth=2, label=f'Q = {Q} m³/s')
ax.axhline(y=h_n, color='g', linestyle='--', linewidth=2, label=f'h_n = {h_n:.3f} m')
ax.plot(Q, h_n, 'ko', markersize=10, label='Ponto normal')

ax.set_xlabel('Vazão [m³/s]', fontsize=12)
ax.set_ylabel('Profundidade [m]', fontsize=12)
ax.set_title('Curva Vazão-Profundidade - Canal Trapezoidal', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Interpretação:")
print("  • A curva é não-linear (cresce mais rápido que linear)")
print("  • Para Q fixo, existe uma única h_n (solução única)")
print("  • Aumentar Q exige aumento de h (mais que proporcional)")

---

## 🔧 Seção 5: Análise de Sensibilidade

Vamos analisar como $h_n$ varia com os parâmetros do canal:
- Rugosidade $n$
- Declividade $S_0$
- Vazão $Q$

In [ ]:
# ============================================================
# ANÁLISE DE SENSIBILIDADE
# ============================================================

print("=" * 60)
print("ANÁLISE DE SENSIBILIDADE")
print("=" * 60)

# Função para calcular h_n
def calcular_hn(b, m, n, S0, Q):
    def f(h):
        return Q_Manning(b, m, n, S0, h) - Q
    return brentq(f, 0.01, 10.0)

# Sensibilidade a n
n_range = np.linspace(0.015, 0.050, 10)
hn_n = [calcular_hn(b, m, n_val, S0, Q) for n_val in n_range]

print(f"\n📊 Sensibilidade à rugosidade n:")
print(f"{'n':<10} {'h_n [m]':<12} {'Δh_n [%]':<12}")
print("-" * 35)
for n_val, h_val in zip(n_range, hn_n):
    delta_h = (h_val - h_n) / h_n * 100
    print(f"{n_val:<10.3f} {h_val:<12.4f} {delta_h:<+12.1f}")

# Sensibilidade a S0
S0_range = np.logspace(-4, -2, 10)
hn_S0 = [calcular_hn(b, m, n, S0_val, Q) for S0_val in S0_range]

print(f"\n📊 Sensibilidade à declividade S₀:")
print(f"{'S₀':<12} {'h_n [m]':<12} {'Δh_n [%]':<12}")
print("-" * 35)
for S0_val, h_val in zip(S0_range, hn_S0):
    delta_h = (h_val - h_n) / h_n * 100
    print(f"{S0_val:<12.1e} {h_val:<12.4f} {delta_h:<+12.1f}")

# Sensibilidade a Q
Q_range = np.linspace(1.0, 15.0, 10)
hn_Q = [calcular_hn(b, m, n, S0, Q_val) for Q_val in Q_range]

print(f"\n📊 Sensibilidade à vazão Q:")
print(f"{'Q [m³/s]':<12} {'h_n [m]':<12} {'Δh_n [%]':<12}")
print("-" * 35)
for Q_val, h_val in zip(Q_range, hn_Q):
    delta_h = (h_val - h_n) / h_n * 100
    print(f"{Q_val:<12.1f} {h_val:<12.4f} {delta_h:<+12.1f}")

---

## 💡 Seção 6: Discussão e Interpretação

### Resultados Principais

1. **Profundidade normal:** $h_n \approx 1.2$ m para os parâmetros dados.

2. **Sensibilidade à rugosidade:** Aumentar $n$ de 0.025 para 0.050 (100%) aumenta $h_n$ em ~30%. Canais mais rugosos exigem maior profundidade para transportar a mesma vazão.

3. **Sensibilidade à declividade:** Aumentar $S_0$ de 0.001 para 0.01 (10×) reduz $h_n$ em ~40%. Canais mais íngremes transportam mais vazão com menor profundidade.

4. **Sensibilidade à vazão:** A relação $Q \times h_n$ é não-linear. Dobrar $Q$ aumenta $h_n$ em menos de 100% (tipicamente 60-80%).

### Aplicações Práticas

- **Dimensionamento de canais:** Conhecendo $Q$, $n$ e $S_0$, calculamos $h_n$ para definir a altura das margens (com borda livre de segurança).

- **Análise de capacidade:** Se $h_{\text{máx}} < h_n$, o canal não tem capacidade suficiente e transbordará.

- **Otimização de seção:** Para minimizar área molhada (e custo de escavação), busca-se a seção hidráulica ótima.

### Limitações

- **Regime permanente:** A equação de Manning assume escoamento uniforme e permanente. Para escoamentos transientes (cheias), são necessárias as equações de Saint-Venant completas.

- **Rugosidade constante:** Na prática, $n$ varia com a profundidade, vegetação e sedimentos. Modelos avançados consideram $n(h)$.

- **Seção constante:** Canais naturais têm geometria variável ao longo do comprimento. Modelos 1D com seção transversal variável são mais realistas.

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 5](../notebooks/05_Hidrodinamica_Canais_Abertos.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 5.2**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>